In [2]:
import yfinance as yf
from fredapi import Fred
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [4]:
#stock_symbol=input(f"Enter stock symbol (ex: AAPL,TSLA): ").upper()
stock_symbol="AAPL"
default_startdate=(datetime.today()-timedelta(days=5*365)).strftime("%Y-%m-%d")
default_enddate=datetime.today().strftime("%Y-%m-%d")
startdate=input(f"enter the startdate in (YYYY-MM-DD) [default date is {default_startdate}]: ") or default_startdate
enddate=input(f"enter the enddate in (YYYY-MM-DD) [default date is {default_enddate}]: " ) or default_enddate
stockdata=yf.download(stock_symbol,start=startdate, end=enddate,group_by="column")
# Check if MultiIndex exists and flatten it
if isinstance(stockdata.columns, pd.MultiIndex):
    stockdata.columns = stockdata.columns.droplevel(1)  # Drop first-level index
if "Price" in stockdata.columns:
    stockdata=stockdata.drop(columns=["Price"])
stockdata=stockdata.reset_index() #makes date column the first column by creating a new default integer index column.
stockdata.columns=["Date","Close","High","Low","Open","Volume"]
stockdata.head()

              
                                                          

enter the startdate in (YYYY-MM-DD) [default date is 2020-02-24]:  
enter the enddate in (YYYY-MM-DD) [default date is 2025-02-22]:  


[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
0,2020-02-24,72.362038,73.818112,70.190065,72.138777,222195200
1,2020-02-25,69.910973,73.417688,69.437753,73.034258,230673600
2,2020-02-26,71.020012,72.289226,69.527537,69.534818,198054800
3,2020-02-27,66.377571,69.406208,66.241671,68.217082,320605600
4,2020-02-28,66.338753,67.564287,62.215637,62.431625,426510000


In [5]:
stockdata.count()

Date      1257
Close     1257
High      1257
Low       1257
Open      1257
Volume    1257
dtype: int64

In [6]:
stockdata.shape

(1257, 6)

In [7]:
stockdata.describe()

,Date,Close,High,Low,Open,Volume
count,1257,1257.000000,1257.000000,1257.000000,1257.000000,1.257000e+03
mean,2022-08-20 18:57:33.937947648,156.538234,158.120314,154.775791,156.379978,8.847017e+07
min,2020-02-24 00:00:00,54.449894,55.452160,51.595990,55.350235,2.323470e+07
25%,2021-05-21 00:00:00,129.967834,131.443316,128.397138,129.870028,5.414710e+07
50%,2022-08-19 00:00:00,154.073456,155.747753,151.672010,153.521798,7.440380e+07
75%,2023-11-17 00:00:00,180.856033,182.209631,179.283467,181.003268,1.034332e+08
max,2025-02-21 00:00:00,258.735504,259.814335,257.347047,257.906429,4.265100e+08
std,NaN,42.205974,42.343822,41.972323,42.139532,5.301822e+07


In [8]:
print(stockdata.isna().sum())
print(f"duplicated values: {stockdata[stockdata.duplicated]}")

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64
duplicated values: Empty DataFrame
Columns: [Date, Close, High, Low, Open, Volume]
Index: []


In [9]:
stockdata.dtypes


Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

Now lets add more technical indicators in yfinance dataset
1) Simple Moving Average(SMA):SMA calculates the average of closing price over a specific window.here we have used a default window size of 20.
if the current price in a specific row is greater than SMA then its upward trend and vice versa.
2) Exponential Moving Average (EMA): The EMA is a type of moving average that places exponentially decreasing weights on older data points. In other words, recent prices affect the EMA more than prices from the distant past.
Multiplier= 2/N+1, N=window size.
EMA(today)=(price(today)-EMA(yesterday))*multiplier+EMA(yesterday)
Comparing EMA to the Current Price:
If the current price is above the EMA, it often signals bullish sentiment (an upward trend), while a price below the EMA may indicate a bearish trend. This method provides a direct check of whether current market action is above or below the smoothed trend.
Observing the Direction of the EMA Itself:
An increasing (rising) EMA over time suggests that the average price is trending upward, which is another indication of an upward trend. Conversely, a falling EMA indicates a downward trend.
Caution: If the price is above the EMA but the EMA is turning down (or vice versa), it may signal a potential change in trend.
3) Relative Strength Index (RSI):
The RSI is a momentum indicator that oscillates between 0 and 100. It helps traders identify overbought or oversold conditions in a market.
Gains: When the current price is higher than the previous price.
Losses: When the current price is lower than the previous price.
Over a typical period (commonly 14 days), calculate:
Average Gain: Sum of gains over the period divided by the number of periods.
Average Loss: Sum of losses over the period divided by the number of periods.
The RS is then calculated as:
RS=Average Gain/Average Loss
RSI=1-1/100+RS (to convert range between 0 and 100)
Imagine a Scale from 0 to 100:
The RSI gives you a number between 0 and 100. This number is a quick way to see if the stock is being bought or sold too aggressively.
High RSI (Above 70):
When the RSI is high—say, above 70—it means the stock has been bought a lot recently. In simple terms, the market might be overly enthusiastic about the stock, pushing the price too high. This situation is often called "overbought" and might suggest that the stock could be due for a pullback or a drop soon.
Low RSI (Below 30):
When the RSI is low—below 30—it indicates the stock has been sold off a lot. Here, the market might be too pessimistic, and the stock might be "oversold." This condition could mean that the stock is undervalued at the moment and might bounce back.
Middle Range (Between 30 and 70):
If the RSI is in the middle, it usually means the market is relatively balanced. The stock isn't showing extreme behavior, so there’s no strong signal that it’s overbought or oversold.
4) Moving Average Convergence Divergence (MACD):
The MACD shows the relationship between two exponential moving averages (EMAs) of a stock's price. It helps you see how quickly the price is moving and whether that momentum is increasing or decreasing.
MACD Line:
This is calculated by subtracting a longer-term EMA (typically the 26-day EMA) from a shorter-term EMA (typically the 12-day EMA).
In simple terms: If the 12-day EMA is much higher than the 26-day EMA, it shows that recent prices are rising faster compared to the longer-term trend.
Signal Line: (takes most recent 9 values)
This is usually a 9-day EMA of the MACD line.
Purpose: It smooths out the MACD line and helps you identify changes in momentum.
Histogram:
The histogram represents the difference between the MACD line and the signal line.
What it tells you: A growing histogram indicates that the gap between the MACD line and the signal line is widening, which often means the momentum is strengthening in one direction.
How to Interpret the MACD
Bullish Signal:
If the MACD line crosses above the signal line, it may indicate that the stock is picking up momentum and that it might be a good time to buy.
Bearish Signal:
If the MACD line crosses below the signal line, it can signal that the momentum is slowing or reversing, suggesting a potential sell signal.
Histogram Insights:
The size of the histogram bars tells you how strong the momentum is. Bigger bars mean stronger momentum, while smaller bars indicate that the momentum is weakening.
5) Bollinger Bands (Upper and Lower Bands):
Bollinger Bands are a set of three lines plotted on a price chart that help you understand both the trend and the volatility of a stock. They consist of:
Middle Band:
Typically a Simple Moving Average (SMA) over a set number of periods (commonly 20 days).
Upper Band:
Calculated by taking the middle band and adding two standard deviations of the price. This band shows the upper boundary of the price movement.
Lower Band:
Calculated by taking the middle band and subtracting two standard deviations of the price. This band shows the lower boundary of the price movement.
How They Work
Dynamic Boundaries:
The upper and lower bands move based on the stock’s volatility. When the market is volatile, the standard deviation increases, and the bands widen. When the market is less volatile, the bands contract.
Volatility Indicator:
The widening or narrowing of the bands can signal changes in market volatility. Wide bands indicate high volatility, and narrow bands indicate low volatility.
Interpreting Price Action:
Price near Upper Band: May suggest the stock is overbought or has been in a strong upward trend.
Price near Lower Band: May indicate the stock is oversold or in a downward trend.
Price Crossing Bands: A move that touches or breaks through a band can sometimes signal a potential reversal or continuation, but this should be confirmed with other indicators.

6) Average True Range (ATR):
ATR tells you, on average, how much the stock's price moves in a day (or another time period). It doesn’t say if the price is going up or down, only how "wild" or "calm" the price action is.
Volatility Measure:
A higher ATR means the stock is experiencing larger price swings (high volatility). A lower ATR indicates smaller price movements (low volatility).
True Range (TR):
For each day, the True Range is determined by taking the greatest value among:
Current Day’s Range: The difference between the day’s high and low.
Gap Up: The absolute difference between the day’s high and the previous day’s close.
Gap Down: The absolute difference between the day’s low and the previous day’s close.
This ensures that sudden gaps in price from one day to the next are captured.
Averaging the True Range:
Once you have the True Range for each day, the ATR is computed by averaging these values over a set number of days (commonly 14). This average smooths out day-to-day fluctuations and gives you a clear picture of the typical price movement.
In simple terms, ATR helps you gauge the "nervousness" of the stock. High values suggest you might need to be extra cautious because prices are moving a lot, while low values indicate a more stable, predictable environment.
7) On-Balance Volume (OBV)
On-Balance Volume (OBV) is a momentum indicator that uses volume flow to predict changes in stock price. In simple terms, it helps you understand whether buyers or sellers are in control by tracking how much volume is moving in and out of a stock over time.
How OBV Is Calculated
Starting Point:
OBV starts at a baseline value, often zero.
Daily Changes:
For each trading day, you adjust the OBV based on the stock’s closing price:
If today's closing price is higher than yesterday's:
Add the day's volume to the previous OBV.
If today's closing price is lower than yesterday's:
Subtract the day's volume from the previous OBV.
If the closing price is unchanged:
The OBV remains the same.
Cumulative Effect:
This process creates a running total that reflects the net flow of volume. It builds up over time, indicating whether more volume is coming in on up days (suggesting accumulation) or down days (suggesting distribution).
What OBV Tells You
Rising OBV:
A steadily increasing OBV means that buying pressure is accumulating—even if the price hasn't yet risen dramatically. This can be a signal that the stock may be gearing up for an upward move.
Falling OBV:
Conversely, a declining OBV indicates that selling pressure is prevailing. Even if the price is temporarily holding steady, the consistent outflow of volume might signal a forthcoming drop in price.
Divergence:
Sometimes, the OBV might move in a different direction than the price:
Bullish Divergence: If the price is falling but OBV is rising, it might indicate hidden buying strength.
Bearish Divergence: If the price is rising but OBV is falling, it could mean that the upward move is not supported by volume and may soon reverse.

In [17]:



# Assuming 'stockdata' is your DataFrame with columns: 'Open', 'High', 'Low', 'Close', 'Volume'

# 1. Simple Moving Average (SMA) - 20-day
stockdata['SMA_20'] = stockdata['Close'].rolling(window=20, min_periods=1).mean()

# 2. Exponential Moving Average (EMA) - 20-day
stockdata['EMA_20'] = stockdata['Close'].ewm(span=20, adjust=False).mean()

# 3. Relative Strength Index (RSI) - 14-day
def compute_RSI(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    # Use exponential moving average for a smoother average gain/loss
    avg_gain = gain.ewm(alpha=1/period, min_periods=1, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=1, adjust=False).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

stockdata['RSI_14'] = compute_RSI(stockdata['Close'], period=14)

# 4. Moving Average Convergence Divergence (MACD)
# Compute 12-day EMA and 26-day EMA first:
stockdata['EMA_12'] = stockdata['Close'].ewm(span=12, adjust=False).mean()
stockdata['EMA_26'] = stockdata['Close'].ewm(span=26, adjust=False).mean()
stockdata['MACD'] = stockdata['EMA_12'] - stockdata['EMA_26']
# Compute the 9-day EMA of MACD as the signal line:
stockdata['MACD_Signal'] = stockdata['MACD'].ewm(span=9, adjust=False).mean()
stockdata['MACD_Hist'] = stockdata['MACD'] - stockdata['MACD_Signal']

# 5. Bollinger Bands (20-day SMA ± 2 standard deviations)
rolling_std = stockdata['Close'].rolling(window=20, min_periods=1).std(ddof=0)
stockdata['Bollinger_Middle'] = stockdata['SMA_20']  # same as SMA_20
stockdata['Bollinger_Upper'] = stockdata['SMA_20'] + (2 * rolling_std)
stockdata['Bollinger_Lower'] = stockdata['SMA_20'] - (2 * rolling_std)

# 6. Average True Range (ATR) - 14-day
def compute_ATR(df, period=14):
    high = df['High']
    low = df['Low']
    close = df['Close']
    tr1 = high - low
    tr2 = (high - close.shift()).abs()
    tr3 = (low - close.shift()).abs()
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = true_range.rolling(window=period, min_periods=1).mean()
    return atr

stockdata['ATR_14'] = compute_ATR(stockdata, period=14)

# 7. On-Balance Volume (OBV)
def compute_OBV(df):
    obv = [0]  # starting OBV value
    for i in range(1, len(df)):
        if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
            obv.append(obv[-1] + df['Volume'].iloc[i])
        elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
            obv.append(obv[-1] - df['Volume'].iloc[i])
        else:
            obv.append(obv[-1])
    return pd.Series(obv, index=df.index)

stockdata['OBV'] = compute_OBV(stockdata)

# If any remaining NaN values exist, use linear interpolation to fill them.
stockdata.interpolate(method='linear', inplace=True)

# Optionally, inspect the first few rows of the DataFrame
print(stockdata.head())



        Date      Close       High        Low       Open     Volume  \
0 2020-02-24  72.362038  73.818112  70.190065  72.138777  222195200   
1 2020-02-25  69.910973  73.417688  69.437753  73.034258  230673600   
2 2020-02-26  71.020012  72.289226  69.527537  69.534818  198054800   
3 2020-02-27  66.377571  69.406208  66.241671  68.217082  320605600   
4 2020-02-28  66.338753  67.564287  62.215637  62.431625  426510000   

      SMA_20     EMA_20     RSI_14     EMA_12     EMA_26      MACD  \
0  72.362038  72.362038        NaN  72.362038  72.362038  0.000000   
1  71.136505  72.128603   0.000000  71.984951  72.180477 -0.195527   
2  71.097674  72.023023  32.763073  71.836499  72.094517 -0.258018   
3  69.917648  71.485361  13.227130  70.996664  71.671039 -0.674376   
4  69.201869  70.995208  13.156488  70.280062  71.276055 -0.995993   

   MACD_Signal  MACD_Hist  Bollinger_Middle  Bollinger_Upper  Bollinger_Lower  \
0     0.000000   0.000000         72.362038        72.362038        72.

In [19]:
for col in stockdata:
    print(f"column {col} with number of missing values is : {stockdata[col].isna().sum()}")

column Date with number of missing values is : 0
column Close with number of missing values is : 0
column High with number of missing values is : 0
column Low with number of missing values is : 0
column Open with number of missing values is : 0
column Volume with number of missing values is : 0
column SMA_20 with number of missing values is : 0
column EMA_20 with number of missing values is : 0
column RSI_14 with number of missing values is : 1
column EMA_12 with number of missing values is : 0
column EMA_26 with number of missing values is : 0
column MACD with number of missing values is : 0
column MACD_Signal with number of missing values is : 0
column MACD_Hist with number of missing values is : 0
column Bollinger_Middle with number of missing values is : 0
column Bollinger_Upper with number of missing values is : 0
column Bollinger_Lower with number of missing values is : 0
column ATR_14 with number of missing values is : 0
column OBV with number of missing values is : 0


1) CPIAUCSL – Consumer Price Index: it measures how much prices of everyday items are increasing or decreasing every day. its an indicator of inflation.Rise in cpi mean cost of living is going up meaning inflation is happening. When cpi rises too much federal reserve raises interest rates to slow down inflation.
CPI rising too fast(>5%) High inflation -> stocks fall.
CPI falling from high levels(5%-2%) -> stocks rise
CPI near zero or negative(deflation) -> stocks crash(weak economy)

2) FEDFUNDS – Effective Federal Funds Rate: 
Banks need to maintain certain amount of cash. if it doesnt have enough cash, it borrows from another bank. The interest rate they pay for this loan is called the federal funds rate.
Federal reserve controls this rate and adjusts it to influence the economy.
High Inflation -> Fed raises rates, stock prises fall.
Fed lowers interest rates -> loans become cheaper->demand increases->stock prices rise.
If FEDFUNDS is rising → Expect stocks to fall 📉 (expensive loans, less spending).
If FEDFUNDS is stable or falling → Expect stocks to rise 📈 (cheap loans, more spending).
If FEDFUNDS is very low (<1%) → Stocks may boom.

3) UNRATE – Unemployment Rate: If unemployment is rising → Expect stocks to fall 📉 (people spend less).
If unemployment is falling → Expect stocks to rise 📈 (people spend more).
If unemployment is very low (<3%) → Inflation risk increases → The Fed may raise rates, which hurts stocks 📉.

4) GDP – Gross Domestic Product:
If GDP is growing (+2% or more) → Stocks rise 📈 (economy is expanding, strong profits).
If GDP is slowing down (+0% to +2%) → Stocks may stay stable or move sideways.
If GDP is negative (recession, <0%) → Stocks fall 📉 (weak economy, job losses, panic).

5) VIXCLS – CBOE Volatility Index: Close
If investors expect a stock market crash, they buy put options (bets that stocks will fall).
	•	If many people buy protection against losses, option prices increase, and the VIX goes up.
	•	If investors feel confident, they buy fewer put options, and the VIX goes down.
VIX Value	Market Condition	              Stock Market Impact
Below 15	Low volatility,                    calm market	📈 Stocks likely to rise slowly
15 - 25	    Normal volatility	                   🔄 Stocks fluctuate normally
Above 25	High volatility, fear rising	   📉 Stocks may fall or move unpredictably
Above 40	Extreme panic (crisis level)	   📉📉 Stocks likely to crash

6) SKEW – CBOE SKEW Index
The CBOE SKEW Index measures the likelihood of an extreme stock market crash or big unexpected moves. It tells us how much investors are preparing for a “black swan” event (a sudden, rare market crash).

Unlike the VIX, which measures expected volatility, the SKEW Index measures tail risk—the risk of rare, extreme market events that investors aren’t normally expecting.
If investors think the stock market will crash suddenly, they buy deep out-of-the-money put options (which only pay off if stocks crash hard).
	•	If many people buy this insurance, the SKEW Index goes up.
	•	If no one is worried about a crash, the SKEW Index stays low.
SKEW Value	Market Condition	                                  Stock Market Impact
100 - 115	Market is stable,no big crashes expected	           📈 Stocks likely to rise
115 - 130	Some fear of a rare crash	                           🔄 Stocks may be slightly volatile
130 - 150	High fear of a rare market crash	                   📉 Stocks may face a sharp drop

7) TEDRATE – TED Spread (Difference Between 3-Month Treasury Bill and LIBOR Rate)
   The TED Spread is the difference between:
LIBOR (London Interbank Offered Rate) – The interest rate banks charge each other for short-term loans.
3-Month U.S. Treasury Bill (TB3MS) – The interest rate the U.S. government pays on short-term borrowing.

📌 Formula:

\text{TED Spread} = \text{LIBOR} - \text{3-Month Treasury Bill Rate (TB3MS)}
LIBOR measures risk in the banking system. If banks trust each other, LIBOR stays low.
	•	U.S. Treasury Bills (TB3MS) are risk-free because the U.S. government always repays its debt.
	•	When banks start worrying about a financial crisis, they charge higher interest rates on loans to each other, and LIBOR rises.
	•	This widens the TED Spread, signaling higher financial risk.

💡 Interpreting the TED Spread Levels:
TED SpreadValue	     Market Condition	               Stock Market Impact
Below 0.5%	         Normal, stable banking system	   📈 Stocks stable/rising
0.5% - 1%	         Some stress in banking	           🔄 Possible volatility in stocks
Above 1%	         High financial risk	           📉 Stocks may fall
Above 2%	         Major financial crisis warning    📉📉 Stocks could crash  
   
8) DGS10 – 10-Year Treasury Constant Maturity Rate
The 10-Year Treasury Yield (DGS10) is the interest rate that investors earn when they buy a 10-year U.S. government bond. It is one of the most important indicators in finance because it affects everything from mortgage rates to stock prices.
	•	When DGS10 rises → Investors shift money away from stocks into bonds, and stocks fall 📉.
	•	When DGS10 falls → Investors move money into stocks, and stock prices rise 📈.
The DGS10 competes with stocks for investor money.

✔️ When DGS10 rises (above 3-4%)
	•	Bonds become more attractive than stocks.
	•	Investors sell stocks and buy bonds.
	•	📉 Stock prices fall.

✔️ When DGS10 falls (below 2-3%)
	•	Bonds become less attractive, so investors buy more stocks.
	•	📈 Stock prices rise.
If DGS10 is rising → Investors move to bonds → Stocks fall 📉.
If DGS10 is falling → Investors buy stocks → Stocks rise 📈.
If DGS10 is stable → No major impact on stocks.

9) T10Y3M – 10-Year Treasury Constant Maturity Minus 3-Month Treasury Constant Maturity (Yield Curve Indicator)
The T10Y3M (10-Year Treasury Yield minus 3-Month Treasury Yield) is one of the most powerful recession predictors in the financial world. It is known as the Yield Curve Indicator.
	•	When T10Y3M is positive → Normal economy, stocks rise 📈.
	•	When T10Y3M is negative (Inverted Yield Curve) → Recession warning, stocks fall 📉.
The 10-Year Treasury Yield (DGS10) is what investors earn for lending money to the U.S. government for 10 years.
	•	The 3-Month Treasury Yield (TB3MS) is what investors earn for lending money for just 3 months.
	•	Normally, long-term interest rates (10-year) should be higher than short-term rates (3-month) because investors expect higher returns for locking money away longer.
	•	But sometimes, short-term rates become higher than long-term rates, creating an “inverted yield curve”, which is a strong recession warning.
If T10Y3M is positive (>0%) → Economy is normal, stocks rise 📈.
If T10Y3M is near 0% → Market volatility increases, stocks become unstable 🔄.
If T10Y3M is negative (<0%) → Recession risk high, stocks fall 📉.

The Fed sets the Federal Funds Rate, which directly influences short-term interest rates (like the 3-month Treasury yield).
✔️ When inflation rises too high, the Fed raises short-term interest rates to slow down the economy.
✔️ Higher short-term rates mean banks charge more for loans, which reduces borrowing and spending.
When investors expect a recession, they become fearful and move their money into long-term Treasury bonds (10-year bonds) because they are safer than stocks.
	•	When many investors buy long-term bonds, the demand increases, which causes the price of bonds to go up.
	•	When bond prices go up, yields go down (bond yields and bond prices move in opposite directions).
	•	This means that even though the Fed is raising short-term rates, long-term bond yields stay low or even drop.
final note:during recessiion fear fed raises interest rates short term interets and it goes up and investers prefer bonds due to which long term yield decreases and signals recession

10) M2SL – M2 Money Stock (Total Money Supply in the Economy)
M2 includes all the money that people and businesses can quickly use to buy things or invest. It consists of:

✔️ M1 (Cash & Checking Deposits): Money people can use immediately.
✔️ Savings Accounts & Money Market Accounts: Money that earns interest but is still accessible.
✔️ Small-Time Deposits (<$100,000 CDs): Fixed deposits that can be withdrawn easily.

💡 M2 does NOT include large institutional deposits or money that is locked in long-term investments.
1️⃣ If M2 is increasing → More liquidity in the economy → Stocks rise 📈.
2️⃣ If M2 is stable → Neutral effect on stocks 🔄.
3️⃣ If M2 is decreasing → Less money in the economy → Stocks fall 📉.
4️⃣ If M2 shrinks rapidly → Recession risk increases 🚨.

11) INDPRO – Industrial Production Index (Total Production of Factories, Mining, and Utilities in the U.S.)
The Industrial Production Index (INDPRO) measures the total output of factories, mining, and utilities in the U.S. It tells us how much physical goods are being produced, making it a key indicator of economic strength.
	•	When INDPRO increases → More production, strong economy → Stocks rise 📈.
	•	When INDPRO decreases → Less production, weak economy → Stocks fall 📉.
1️⃣ If INDPRO is rising (+2% or more) → Economy is strong, stocks rise 📈.
2️⃣ If INDPRO is flat (0% to +2%) → No major impact, stocks stay stable 🔄.
3️⃣ If INDPRO is falling (<0%) → Economy slowing down, stocks may fall 📉.
4️⃣ If INDPRO is falling fast (>5% drop) → Recession risk, stocks may crash 🚨.

12) UMCSENT – University of Michigan: Consumer Sentiment Index
Surveys 500+ households every month to ask:
	•	“Do you think the economy is getting better or worse?”
	•	“Are you planning to buy big items (cars, homes, appliances)?”
	•	“Do you feel financially secure?”
✔️ The answers create an index score from 0 to 100+.
✔️ Higher scores = More confidence → More spending.
✔️ Lower scores = Less confidence → Less spending.

📌 Example:
	•	If UMCSENT = 90, most people feel positive about their finances → Higher spending → Stock market rises.
	•	If UMCSENT = 60, people worry about job security and inflation → Lower spending → Stock market falls.
1️⃣ If UMCSENT is above 80 → Strong economy, people spending → Stocks rise 📈.
2️⃣ If UMCSENT is between 60-80 → Uncertainty in the economy → Stocks volatile 🔄.
3️⃣ If UMCSENT is below 60 → Consumers are cutting spending → Stocks may fall 📉.
4️⃣ If UMCSENT is below 50 → Recession risk, stocks may crash 🚨.

In [20]:
#Fetch all the above features from the FRED(Federal Reserve Economic Data) dataset
#Define fred indiators and their description
series_ids={'CPIAUCSL': 'Consumer Price Index (Inflation)',
    'FEDFUNDS': 'Federal Funds Rate (Interest Rate)',
    'UNRATE': 'Unemployment Rate',
    'GDP': 'Gross Domestic Product (GDP)',
    'VIXCLS': 'CBOE Volatility Index (Market Volatility)',
    'TEDRATE': 'TED Spread (Financial Stress)',
    'DGS10': '10-Year Treasury Yield (Bond Market)',
    'T10Y3M': 'Yield Curve (Recession Predictor)',
    'M2SL': 'M2 Money Stock (Total Money Supply)',
    'INDPRO': 'Industrial Production Index (Economic Output)',
    'UMCSENT': 'University of Michigan Consumer Sentiment Index'
}
# Initialize FRED API
API_KEY = "d8b18a1fd044f9ef19746f47a6ffc6f1"
fred = Fred(api_key=API_KEY)
fred_data={}
for series_id, description in series_ids.items():
    try:
        fred_data[series_id] = fred.get_series(series_id,startdate,enddate)
    except Exception as e:
        print(f"Error fetching {series_id}: {e}")
df_fred=pd.DataFrame(fred_data)
df_fred.index=pd.to_datetime(df_fred.index)
print(df_fred.head())


            CPIAUCSL  FEDFUNDS  UNRATE        GDP  VIXCLS  TEDRATE  DGS10  \
2020-01-01       NaN       NaN     NaN  21727.657     NaN      NaN    NaN   
2020-02-01    259.25      1.58     3.5        NaN     NaN      NaN    NaN   
2020-02-24       NaN       NaN     NaN        NaN   25.03     0.15   1.38   
2020-02-25       NaN       NaN     NaN        NaN   27.85     0.14   1.33   
2020-02-26       NaN       NaN     NaN        NaN   27.56     0.11   1.33   

            T10Y3M     M2SL    INDPRO  UMCSENT  
2020-01-01     NaN      NaN       NaN      NaN  
2020-02-01     NaN  15453.8  101.6718    101.0  
2020-02-24   -0.15      NaN       NaN      NaN  
2020-02-25   -0.20      NaN       NaN      NaN  
2020-02-26   -0.20      NaN       NaN      NaN  


now fill missing values based on update frequency
1) FRED Code	Update Frequency
2) CPIAUCSL	Monthly
3) FEDFUNDS	Daily
4) UNRATE	Monthly
5) GDP	Quarterly
6) VIXCLS	Daily
7) TEDRATE	Daily
8) DGS10	Daily
9) T10Y3M	Daily
10) M2SL	Weekly
11) INDPRO	Monthly
12) UMCSENT	Monthly


In [21]:
# Interpolate daily-changing values (smooth trends)
for col in ['FEDFUNDS', 'DGS10', 'T10Y3M', 'M2SL','TEDRATE']:
    df_fred[col]=df_fred[col].interpolate(method='time')

# Forward fill for monthly/quarterly data
for col in ['CPIAUCSL', 'UNRATE', 'INDPRO', 'UMCSENT', 'GDP', 'M2SL']:
    df_fred[col]=df_fred[col].ffill()

# Backward fill to fill remaining NaNs (especially for early missing values)
df_fred=df_fred.bfill()
df_fred.head()
    

,CPIAUCSL,FEDFUNDS,UNRATE,GDP,VIXCLS,TEDRATE,DGS10,T10Y3M,M2SL,INDPRO,UMCSENT
2020-01-01,259.25,1.580000,3.5,21727.657,25.03,0.15,1.38,-0.15,15453.800000,101.6718,101.0
2020-02-01,259.25,1.580000,3.5,21727.657,25.03,0.15,1.38,-0.15,15453.800000,101.6718,101.0
2020-02-24,259.25,0.842414,3.5,21727.657,25.03,0.15,1.38,-0.15,15871.606897,101.6718,101.0
2020-02-25,259.25,0.810345,3.5,21727.657,27.85,0.14,1.33,-0.20,15889.772414,101.6718,101.0
2020-02-26,259.25,0.778276,3.5,21727.657,27.56,0.11,1.33,-0.20,15907.937931,101.6718,101.0


In [22]:
df_fred.isna().sum()

CPIAUCSL    0
FEDFUNDS    0
UNRATE      0
GDP         0
VIXCLS      1
TEDRATE     0
DGS10       0
T10Y3M      0
M2SL        0
INDPRO      0
UMCSENT     0
dtype: int64

In [23]:
df_fred.dropna(how='all', inplace=True)

In [24]:
df_fred.isna().sum()

CPIAUCSL    0
FEDFUNDS    0
UNRATE      0
GDP         0
VIXCLS      1
TEDRATE     0
DGS10       0
T10Y3M      0
M2SL        0
INDPRO      0
UMCSENT     0
dtype: int64

In [25]:
duplicate_rows=df_fred[df_fred.duplicated()]
print(len(duplicate_rows))

1



we will remove duplicated if any after merging with yfinance dataset.


In [26]:
df_fred.reset_index(inplace=True)
df_fred.rename(columns={"index":"Date"},inplace=True)

In [27]:
df_fred.head()

,Date,CPIAUCSL,FEDFUNDS,UNRATE,GDP,VIXCLS,TEDRATE,DGS10,T10Y3M,M2SL,INDPRO,UMCSENT
0,2020-01-01,259.25,1.580000,3.5,21727.657,25.03,0.15,1.38,-0.15,15453.800000,101.6718,101.0
1,2020-02-01,259.25,1.580000,3.5,21727.657,25.03,0.15,1.38,-0.15,15453.800000,101.6718,101.0
2,2020-02-24,259.25,0.842414,3.5,21727.657,25.03,0.15,1.38,-0.15,15871.606897,101.6718,101.0
3,2020-02-25,259.25,0.810345,3.5,21727.657,27.85,0.14,1.33,-0.20,15889.772414,101.6718,101.0
4,2020-02-26,259.25,0.778276,3.5,21727.657,27.56,0.11,1.33,-0.20,15907.937931,101.6718,101.0


In [28]:
df_fred=df_fred.iloc[2:].reset_index(drop=True)
df_fred.head()

,Date,CPIAUCSL,FEDFUNDS,UNRATE,GDP,VIXCLS,TEDRATE,DGS10,T10Y3M,M2SL,INDPRO,UMCSENT
0,2020-02-24,259.25,0.842414,3.5,21727.657,25.03,0.15,1.38,-0.15,15871.606897,101.6718,101.0
1,2020-02-25,259.25,0.810345,3.5,21727.657,27.85,0.14,1.33,-0.20,15889.772414,101.6718,101.0
2,2020-02-26,259.25,0.778276,3.5,21727.657,27.56,0.11,1.33,-0.20,15907.937931,101.6718,101.0
3,2020-02-27,259.25,0.746207,3.5,21727.657,39.16,0.16,1.30,-0.15,15926.103448,101.6718,101.0
4,2020-02-28,259.25,0.714138,3.5,21727.657,40.11,0.21,1.13,-0.14,15944.268966,101.6718,101.0


In [29]:
df_fred['Date'].nunique()

1320

In [30]:
stockdata['Date'].nunique()

1257

In [34]:
merged_df = pd.merge(stockdata, df_fred, how='inner', on='Date')
merged_df.head()

,Date,Close,High,Low,Open,Volume,SMA_20,EMA_20,RSI_14,EMA_12,...,FEDFUNDS,UNRATE,GDP,VIXCLS,TEDRATE,DGS10,T10Y3M,M2SL,INDPRO,UMCSENT
0,2020-02-24,72.362038,73.818112,70.190065,72.138777,222195200,72.362038,72.362038,NaN,72.362038,...,0.842414,3.5,21727.657,25.03,0.15,1.38,-0.15,15871.606897,101.6718,101.0
1,2020-02-25,69.910973,73.417688,69.437753,73.034258,230673600,71.136505,72.128603,0.000000,71.984951,...,0.810345,3.5,21727.657,27.85,0.14,1.33,-0.20,15889.772414,101.6718,101.0
2,2020-02-26,71.020012,72.289226,69.527537,69.534818,198054800,71.097674,72.023023,32.763073,71.836499,...,0.778276,3.5,21727.657,27.56,0.11,1.33,-0.20,15907.937931,101.6718,101.0
3,2020-02-27,66.377571,69.406208,66.241671,68.217082,320605600,69.917648,71.485361,13.227130,70.996664,...,0.746207,3.5,21727.657,39.16,0.16,1.30,-0.15,15926.103448,101.6718,101.0
4,2020-02-28,66.338753,67.564287,62.215637,62.431625,426510000,69.201869,70.995208,13.156488,70.280062,...,0.714138,3.5,21727.657,40.11,0.21,1.13,-0.14,15944.268966,101.6718,101.0


In [35]:
merged_df.isna().sum()

Date                0
Close               0
High                0
Low                 0
Open                0
Volume              0
SMA_20              0
EMA_20              0
RSI_14              1
EMA_12              0
EMA_26              0
MACD                0
MACD_Signal         0
MACD_Hist           0
Bollinger_Middle    0
Bollinger_Upper     0
Bollinger_Lower     0
ATR_14              0
OBV                 0
CPIAUCSL            0
FEDFUNDS            0
UNRATE              0
GDP                 0
VIXCLS              1
TEDRATE             0
DGS10               0
T10Y3M              0
M2SL                0
INDPRO              0
UMCSENT             0
dtype: int64